In [ ]:
def top_amentities(df_small, df_large):
    all_amen = Counter()
    for lst in df_small.index:
        amen = df_large.loc[df_large["listing_id"] == df_small.loc[lst, "listing_id"], "amenity_list"]
        if len(amen):
            all_amen.update(amen.iloc[0])
    top_amenities = [a for a,_ in all_amen.most_common(TOP_AMENITIES)]
    
    for amen in top_amenities:
        df_small[f"amenity__{amen}"] = (
                                        df_large["amenity_list"]
                                        .apply(lambda L: 1 if amen in L else 0)
                                        .values[:len(listings_small)]
                                    )
    return df_small

def encode_host_is_superhost(df):
    df["host_is_superhost_bin"] = listings_small["host_is_superhost"].map({"t":1,"f":0}).fillna(0).astype(int)
    return df


In [ ]:
def vectorize_and_cluster_reviews(reviews):
    print("Building TF-IDF (max_features=%d) ..." % TFIDF_MAX_FEATURES)
    tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=TFIDF_MAX_FEATURES, stop_words="english", min_df=2)
    # fit on all comments
    tfidf_matrix = tfidf.fit_transform(reviews["comments_clean"].fillna("").values)
    print("TF-IDF shape:", tfidf_matrix.shape)
    svd_cols = []
    n_components = min(SVD_COMPONENTS, tfidf_matrix.shape[1]-1)
    print("Applying TruncatedSVD with n_components=", n_components)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    svd_feats = svd.fit_transform(tfidf_matrix)
    svd_cols = [f"svd_text_{i}" for i in range(svd_feats.shape[1])]
    svd_df = pd.DataFrame(svd_feats, columns=svd_cols, index=reviews.index)
    reviews = pd.concat([reviews.reset_index(drop=True), svd_df.reset_index(drop=True)], axis=1)
    
    return reviews